# MultiTaskGP / KroneckerMultiTaskGP: task-aware Bayesian Optimization

関連task間で情報共有する2つの標準modelを比較し、**target taskを固定した候補生成**まで実行します。

- `MultiTaskGP`: long-format。taskごとに異なるXを観測可能
- `KroneckerMultiTaskGP`: block design。同じXですべてのtaskを観測

## このNotebookで確認すること

- task / output / objectiveの違い
- long-formatとblock-designのdata contract
- 両modelのfitとmulti-task posterior
- target taskだけを最適化するacquisition
- task indexをdesign variableとして探索しないcandidate generation
- target taskでの評価と短いBO loop

関連理論:

- [Multi-task / Multi-output GP](../../docs/theory/07_multitask_multioutput.md)
- [Bayesian Optimization](../../docs/theory/01_bayesian_optimization.md)

```text
Related-task data -> MultiTask GP -> target-task posterior
                                  -> acquisition -> optimize x -> evaluate target task
```

この例ではtask 1を最適化対象とします。task IDは関連データ源を表す構造列であり、task 0/1のどちらを選ぶか自体を最適化する設計変数ではありません。

## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf

from robotorchan.models import KroneckerMultiTaskGP, MultiTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### Task・カテゴリ・出力の違い

**Task**は、互いに関連するデータ源や測定対象（例：装置Aと装置B、旧材料と新材料）を表します。**カテゴリ設計変数**は、最適化によって選択する材料種・製法などです。どちらも整数ラベルで表現できますが、問題設定上の役割が異なります。

この例ではtask 0の観測からtask 1の予測を改善することを狙い、**最適化対象はtask 1の応答**と固定します。2つのtaskを同時に最大化する多目的最適化とは異なります。

相関のあるtask間で情報共有できると、目標taskの観測が少ない場合に有利です。ただし、相関が弱い場合には転移による改善が得られないこともあります。

## 2. Related tasks and optimization target

2つの相関したtaskを作ります。BOではtask 1の応答を最大化します。

In [ ]:
def task0(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * x)


def task1(x: torch.Tensor) -> torch.Tensor:
    return 0.7 * torch.sin(2 * torch.pi * x + 0.35) + 0.25


bounds = torch.tensor([[0.0], [1.0]])
target_task = 1

### Long-formatのデータ構造

`MultiTaskGP` の入力は、設計値とtask IDを同じ行に持ちます。例えば `(x=0.2, task=0)` と `(x=0.8, task=1)` は異なる行として保存します。

`train_X_long` は **17行×2列**（設計値、task ID）、`train_Y_long` は **17行×1列**です。taskごとに観測点数や観測位置が違っても扱えます。`task_feature=1` はtask IDがPythonの列番号1にあることを示します。

## 3. MultiTaskGP: long-format training data

task 0とtask 1で異なる観測位置を使います。最後の列はtask identityです。

In [ ]:
x0 = torch.linspace(0.05, 0.95, 10).unsqueeze(-1)
x1 = torch.linspace(0.10, 0.90, 7).unsqueeze(-1)
y0 = task0(x0)
y1 = task1(x1)

train_X_long = torch.cat(
    [
        torch.cat([x0, torch.zeros(len(x0), 1)], dim=-1),
        torch.cat([x1, torch.ones(len(x1), 1)], dim=-1),
    ],
    dim=0,
)
train_Y_long = torch.cat([y0, y1], dim=0)

print("train_X_long:", train_X_long.shape)
print("train_Y_long:", train_Y_long.shape)

## 4. MultiTaskGP construction and fitting

In [ ]:
mt_model = MultiTaskGP(
    train_X=train_X_long,
    train_Y=train_Y_long,
    task_feature=1,
)
mt_model.fit()
mt_model.eval()

print("raw_train_X:", mt_model.raw_train_X.shape)
print("raw_train_Y:", mt_model.raw_train_Y.shape)

### Multi-task posteriorと情報共有

Multi-task GPは、各taskの入力方向の変化とtask間の共分散を同時に学習します。予測時に `output_indices=[0, 1]` を指定すると、2つのtaskについて同じ `test_X` 上の予測を取得できます。

予測平均が近くても不確実性が同じとは限りません。次の追加図では、**観測が少ない目標task 1の事後標準偏差**を可視化します。

## 5. Posterior for both tasks

prediction inputからtask列を省略し、`output_indices`で各taskをoutputとして取得します。

In [ ]:
test_X = torch.linspace(0.0, 1.0, 200).unsqueeze(-1)
with torch.no_grad():
    mt_post = mt_model.posterior(test_X, output_indices=[0, 1])
    mt_mean = mt_post.mean

print("posterior mean shape:", mt_mean.shape)

## 6. MultiTask posterior visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x0.squeeze(-1), y0.squeeze(-1), label="task 0 observations")
ax.scatter(x1.squeeze(-1), y1.squeeze(-1), label="task 1 observations")
ax.plot(test_X.squeeze(-1), mt_mean[..., 0], label="task 0 posterior")
ax.plot(test_X.squeeze(-1), mt_mean[..., 1], label="task 1 posterior")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

### 追加実装例：目標taskの予測区間

モデルが返す `posterior.variance` から標準偏差を計算し、予測平均の周囲に約95%のモデル上の区間を描きます。これは潜在関数の不確実性であり、真の関数が必ず含まれる保証や、測定ノイズを含む予測区間ではありません。

In [ ]:
with torch.no_grad():
    target_std = mt_post.variance[..., 1].clamp_min(0).sqrt()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(test_X.squeeze(-1), mt_mean[..., 1], label="task 1 posterior mean")
ax.fill_between(
    test_X.squeeze(-1),
    mt_mean[..., 1] - 1.96 * target_std,
    mt_mean[..., 1] + 1.96 * target_std,
    alpha=0.2,
    label="approx. 95% latent interval",
)
ax.scatter(x1.squeeze(-1), y1.squeeze(-1), label="task 1 observations")
ax.set_xlabel("x")
ax.set_ylabel("task 1 response")
ax.legend()
plt.show()

## 7. Target-task acquisition

`output_tasks=[1]`のmodelを同じlong-format dataから構築します。これによりposterior/acquisitionの出力をtask 1に限定し、task index自体を探索変数にはしません。

In [ ]:
target_model = MultiTaskGP(
    train_X=train_X_long,
    train_Y=train_Y_long,
    task_feature=1,
    output_tasks=[target_task],
)
target_model.fit()
target_model.eval()

target_observations = train_Y_long[train_X_long[:, 1] == target_task]
acqf = qLogExpectedImprovement(
    model=target_model,
    best_f=target_observations.max(),
)

### なぜtarget taskを固定するのか

`output_tasks=[target_task]` により獲得関数へ渡すモデルの出力をtask 1に限定します。`best_f` にも**task 1の観測値だけ**を使います。task 0の観測は学習に使いますが、task 0の最大値をtask 1の改善基準に混ぜません。

Expected Improvementは $\mathbb E[\max(f_1(x)-f_{1,\mathrm{best}},0)\mid\mathcal D]$ に対応します。候補最適化ではtask IDを探索せず、設計変数 $x$ だけを探索します。

## 8. Candidate generation for task 1

optimizerが探索するのは設計変数`x`だけです。target taskはmodel outputとして固定されています。

In [ ]:
candidate, acq_value = optimize_acqf(
    acq_function=acqf,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)
assert candidate.shape == (1, 1)
assert torch.all((candidate >= bounds[0]) & (candidate <= bounds[1]))
candidate_Y = task1(candidate)

print("candidate x:", candidate)
print("acquisition value:", acq_value)
print("task 1 objective:", candidate_Y)

## 9. Update long-format data

新しいtask 1観測を追加するときだけtask featureを再付与します。

In [ ]:
candidate_long = torch.cat(
    [candidate, torch.full((candidate.shape[0], 1), float(target_task))],
    dim=-1,
)
updated_X_long = torch.cat([train_X_long, candidate_long], dim=0)
updated_Y_long = torch.cat([train_Y_long, candidate_Y], dim=0)

print("updated:", updated_X_long.shape, updated_Y_long.shape)

## 10. Short target-task BO loop

In [ ]:
loop_X = train_X_long.clone()
loop_Y = train_Y_long.clone()
history = []

for iteration in range(3):
    loop_model = MultiTaskGP(
        train_X=loop_X,
        train_Y=loop_Y,
        task_feature=1,
        output_tasks=[target_task],
    )
    loop_model.fit()
    loop_model.eval()

    target_mask = loop_X[:, 1] == target_task
    loop_acqf = qLogExpectedImprovement(
        model=loop_model,
        best_f=loop_Y[target_mask].max(),
    )
    next_X, _ = optimize_acqf(
        acq_function=loop_acqf,
        bounds=bounds,
        q=1,
        num_restarts=5,
        raw_samples=64,
    )
    next_Y = task1(next_X)
    next_X_long = torch.cat(
        [next_X, torch.full((next_X.shape[0], 1), float(target_task))],
        dim=-1,
    )
    history.append((next_X.item(), next_Y.item()))
    loop_X = torch.cat([loop_X, next_X_long], dim=0)
    loop_Y = torch.cat([loop_Y, next_Y], dim=0)

for iteration, (x_value, y_value) in enumerate(history, start=1):
    print(f"iteration={iteration}: x={x_value:.4f}, task1_y={y_value:.4f}")

### Block designとKronecker構造

`KroneckerMultiTaskGP` は、**同じ設計点で全taskを観測する**データを前提とします。入力は `train_X_block: (12, 1)`、出力は `train_Y_block: (12, 2)` です。

Long-formatと違い、task IDを入力列に追加しません。2列の出力がtask 0・1に対応します。新しい設計点を追加する際も両taskを観測する必要があります。片方だけを測る運用にはlong-formatの `MultiTaskGP` が適します。

## 11. KroneckerMultiTaskGP: block-design data

同じ設計点ですべてのtaskを観測できる場合、task identityは`train_Y`のoutput dimensionに置きます。

In [ ]:
train_X_block = torch.linspace(0.05, 0.95, 12).unsqueeze(-1)
train_Y_block = torch.cat(
    [task0(train_X_block), task1(train_X_block)],
    dim=-1,
)

kron_model = KroneckerMultiTaskGP(
    train_X=train_X_block,
    train_Y=train_Y_block,
)
kron_model.fit()
kron_model.eval()

with torch.no_grad():
    kron_post = kron_model.posterior(test_X)

print("train_X_block:", train_X_block.shape)
print("train_Y_block:", train_Y_block.shape)
print("Kronecker posterior:", kron_post.mean.shape)

## 12. Kronecker posterior visualization

In [ ]:
kron_mean = kron_post.mean
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(train_X_block.squeeze(-1), train_Y_block[:, 0], label="task 0 observations")
ax.scatter(train_X_block.squeeze(-1), train_Y_block[:, 1], label="task 1 observations")
ax.plot(test_X.squeeze(-1), kron_mean[..., 0], label="task 0 posterior")
ax.plot(test_X.squeeze(-1), kron_mean[..., 1], label="task 1 posterior")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

## 13. Model selection

| Model | Training representation | Different X by task | Task identity |
|---|---|---:|---|
| `MultiTaskGP` | long format | Yes | `task_feature` in X |
| `KroneckerMultiTaskGP` | block design | No | output dimension of Y |

Kronecker modelもmulti-output posteriorを返します。どのtask/outputをBO objectiveにするかはObjective / PosteriorTransform / acquisition側で明示する必要があり、単に2出力だからmulti-objective BOになるわけではありません。

## 14. Limitations and caveats

- task IDを通常の連続design variableとして最適化しません。
- machineなどのラベルでも、「選択対象」ならcategorical design variable、「関連データ源」ならtaskというように問題設定で意味が変わります。
- task間の関連が弱い場合はnegative transferに注意し、independent modelとも比較します。
- Kroneckerはblock designを要求します。
- このNotebookでは`MultiTaskGP`のtarget-task BOと、`KroneckerMultiTaskGP`の重み付きMC objectiveによる候補生成を両方実行します。

## 15. References

数理、task covariance、MultiTask / ModelList / Kroneckerの使い分けは冒頭のTheory章を参照してください。

### 複数taskの出力から1つの評価指標を作る

Kroneckerモデルは2出力を返します。ここでは $u(x)=0.3f_0(x)+0.7f_1(x)$ という**重み付き単一目的**を定義します。これはPareto最適化ではありません。

`GenericMCObjective` は事後サンプルの最後の出力軸を重み付き和に変換します。`qLogExpectedImprovement` は変換後のスカラー値の改善を評価します。出力の単位やスケールが異なる実務データでは、重みを付ける前に正規化や効用の定義を検討してください。

## 14. Kronecker BO objective: scalarize posterior samples

`KroneckerMultiTaskGP`はmulti-output posteriorを返します。現在のcontractでは`ScalarizedPosteriorTransform`を使ったposterior-level scalarizationは明示的に非対応です。一方、MC acquisitionではposterior samplesに`GenericMCObjective`を適用できます。

ここではtask 0とtask 1を`0.3 / 0.7`で重み付けした単一objectiveを最大化します。これは「task 1だけを選ぶ」例ではなく、両outputから意思決定量を定義する例です。

In [ ]:
from botorch.acquisition.objective import GenericMCObjective
from botorch.sampling.normal import SobolQMCNormalSampler

weights = torch.tensor([0.3, 0.7])
mc_objective = GenericMCObjective(
    lambda samples, X=None: samples @ weights
)
scalarized_observations = train_Y_block @ weights

print("scalarized observations:", scalarized_observations.shape)
print("best scalarized value:", scalarized_observations.max().item())

## 15. Kronecker acquisition and candidate generation

sample-space objectiveを`qLogExpectedImprovement`へ渡し、設計変数`x`だけを最適化します。task軸は`train_Y`のoutput dimensionなのでcandidate Xへtask IDを追加しません。

In [ ]:
kron_acqf = qLogExpectedImprovement(
    model=kron_model,
    best_f=scalarized_observations.max(),
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([64])),
    objective=mc_objective,
)
kron_candidate, kron_acq_value = optimize_acqf(
    acq_function=kron_acqf,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)
assert kron_candidate.shape == (1, 1)
assert torch.all((kron_candidate >= bounds[0]) & (kron_candidate <= bounds[1]))
kron_candidate_Y = torch.cat(
    [task0(kron_candidate), task1(kron_candidate)],
    dim=-1,
)
kron_candidate_objective = kron_candidate_Y @ weights

print("candidate x:", kron_candidate)
print("candidate outputs:", kron_candidate_Y)
print("scalarized objective:", kron_candidate_objective)
print("acquisition value:", kron_acq_value)

## 16. Update the block design

Kroneckerのblock-designでは、新しい`x`を追加するとき同じxですべてのtaskを観測してYの1行を追加します。target taskだけを観測するlong-format更新とは異なります。

In [ ]:
updated_X_block = torch.cat([train_X_block, kron_candidate], dim=0)
updated_Y_block = torch.cat([train_Y_block, kron_candidate_Y], dim=0)

assert updated_X_block.shape[0] == updated_Y_block.shape[0]
assert updated_Y_block.shape[1] == train_Y_block.shape[1]
print("updated block shapes:", updated_X_block.shape, updated_Y_block.shape)

## 17. Short Kronecker BO loop

In [ ]:
kron_loop_X = train_X_block.clone()
kron_loop_Y = train_Y_block.clone()
kron_history = []

for iteration in range(2):
    loop_kron_model = KroneckerMultiTaskGP(
        train_X=kron_loop_X,
        train_Y=kron_loop_Y,
    )
    loop_kron_model.fit()
    loop_kron_model.eval()

    loop_scalarized = kron_loop_Y @ weights
    loop_kron_acqf = qLogExpectedImprovement(
        model=loop_kron_model,
        best_f=loop_scalarized.max(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([64])),
        objective=mc_objective,
    )
    next_X, _ = optimize_acqf(
        acq_function=loop_kron_acqf,
        bounds=bounds,
        q=1,
        num_restarts=5,
        raw_samples=64,
    )
    next_Y = torch.cat([task0(next_X), task1(next_X)], dim=-1)
    next_objective = next_Y @ weights
    kron_history.append((next_X.item(), next_objective.item()))
    kron_loop_X = torch.cat([kron_loop_X, next_X], dim=0)
    kron_loop_Y = torch.cat([kron_loop_Y, next_Y], dim=0)

for iteration, (x_value, objective_value) in enumerate(kron_history, start=1):
    print(
        f"iteration={iteration}: x={x_value:.4f}, "
        f"scalarized_y={objective_value:.4f}"
    )

## 18. Kronecker objective boundary

```text
Kronecker posterior [task 0, task 1]
        ↓ posterior sampling
MC samples [..., q, 2]
        ↓ GenericMCObjective
scalar utility [..., q]
        ↓ qLogEI
candidate x
```

現在のcontractでは、この経路はruntime testで検証されています。一方、`ScalarizedPosteriorTransform`を`KroneckerMultiTaskGP.posterior(...)`へ直接渡す経路はサポート対象ではありません。posterior-level transformとsample-level MC objectiveを同一視しないことが重要です。

## 18. Model selection and caveats

| Model | Training representation | Candidate update |
|---|---|---|
| `MultiTaskGP` | long format | target taskの1観測を追加可能 |
| `KroneckerMultiTaskGP` | block design | candidate xですべてのtaskを追加 |

- weighted scalarizationとmulti-objective BOは別です。
- Pareto frontを直接扱う場合はqLogEHVI/qLogNEHVIなどを使います。
- Kroneckerの利点を維持するにはblock-design observation contractを壊さないことが重要です。
- 一部taskだけを追加観測したい問題ならlong-format `MultiTaskGP`を検討します。

## 19. References

MultiTask / Kroneckerの数理と使い分けは冒頭のTheory章を参照してください。Objective / PosteriorTransformの詳細は後続Phaseで独立して扱います。

### Mixed multi-task・独立モデルの使い分け

- `MixedMultiTaskGP`：カテゴリを設計変数、task IDをデータ源として区別するlong-formatモデル。
- `MixedKroneckerMultiTaskGP`：カテゴリを含む同一設計点で全taskを観測するblock-designモデル。
- `ModelListGP`：独立したGPを組み合わせる方法。task間の共分散を明示的に学習するMulti-task GPとは情報共有の前提が異なります。

次の実装例では、それぞれのモデルで学習と候補生成を確認します。候補のカテゴリは明示的に列挙しており、task IDをカテゴリ設計変数として最適化しているわけではありません。

## 19. Mixed multi-task models and independent ModelListGP

カテゴリは設計変数、taskは出力元の識別子として扱います。MixedMultiTaskGPではtask列を探索せず、MixedKroneckerMultiTaskGPでは同一設計点の両出力を評価します。ModelListGPでは独立GPを結合し、MC objectiveで2出力をスカラー化します。


In [ ]:
from robotorchan.models import (
    MixedKroneckerMultiTaskGP,
    MixedMultiTaskGP,
    ModelListGP,
    SingleTaskGP,
)

mixed_design = torch.cat(
    [
        train_X_block,
        torch.arange(train_X_block.shape[0]).remainder(2)
        .to(train_X_block).unsqueeze(-1),
    ],
    dim=-1,
)
mixed_outputs = torch.cat(
    [
        task0(mixed_design[:, :1]) + 0.1 * mixed_design[:, 1:2],
        task1(mixed_design[:, :1]) + 0.1 * mixed_design[:, 1:2],
    ],
    dim=-1,
)
mixed_long_X = torch.cat(
    [
        torch.cat([mixed_design, torch.zeros(len(mixed_design), 1)], dim=-1),
        torch.cat([mixed_design, torch.ones(len(mixed_design), 1)], dim=-1),
    ],
    dim=0,
)
mixed_long_Y = torch.cat(
    [mixed_outputs[:, :1], mixed_outputs[:, 1:2]], dim=0
)
mixed_mt = MixedMultiTaskGP(
    mixed_long_X, mixed_long_Y, task_feature=2,
    cat_dims=[1], output_tasks=[1],
)
mixed_mt.fit()
mixed_mt.eval()

mixed_kron = MixedKroneckerMultiTaskGP(
    mixed_design, mixed_outputs, cat_dims=[1]
)
mixed_kron.fit()
mixed_kron.eval()

independent_models = ModelListGP(
    SingleTaskGP(train_X_block, train_Y_block[:, :1]),
    SingleTaskGP(train_X_block, train_Y_block[:, 1:2]),
)
fit_gpytorch_mll(independent_models.make_mll())
independent_models.eval()


In [ ]:
mixed_bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])
mixed_target_best = mixed_long_Y[mixed_long_X[:, 2] == 1].max()
mixed_acq = qLogExpectedImprovement(
    model=mixed_mt, best_f=mixed_target_best
)
mixed_choices = []
for category in (0.0, 1.0):
    point, value = optimize_acqf(
        acq_function=mixed_acq,
        bounds=mixed_bounds,
        q=1, num_restarts=2, raw_samples=16,
        fixed_features={1: category},
    )
    mixed_choices.append((value.item(), point))
_, mixed_candidate = max(mixed_choices, key=lambda item: item[0])
mixed_observation = (
    task1(mixed_candidate[:, :1]) + 0.1 * mixed_candidate[:, 1:2]
)
assert mixed_candidate[0, 1].item() in (0.0, 1.0)
print("MixedMultiTaskGP:", mixed_candidate, mixed_observation)

for name, surrogate, training_outputs, candidate_bounds, categorical in [
    ("MixedKroneckerMultiTaskGP", mixed_kron, mixed_outputs, mixed_bounds, True),
    ("ModelListGP", independent_models, train_Y_block, bounds, False),
]:
    best_value = (training_outputs @ weights).max()
    acquisition = qLogExpectedImprovement(
        model=surrogate,
        best_f=best_value,
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([64])),
        objective=mc_objective,
    )
    if categorical:
        choices = []
        for category in (0.0, 1.0):
            point, value = optimize_acqf(
                acq_function=acquisition, bounds=candidate_bounds,
                q=1, num_restarts=2, raw_samples=16,
                fixed_features={1: category},
            )
            choices.append((value.item(), point))
        _, next_point = max(choices, key=lambda item: item[0])
        next_outputs = torch.cat(
            [
                task0(next_point[:, :1]) + 0.1 * next_point[:, 1:2],
                task1(next_point[:, :1]) + 0.1 * next_point[:, 1:2],
            ],
            dim=-1,
        )
    else:
        next_point, _ = optimize_acqf(
            acq_function=acquisition, bounds=candidate_bounds,
            q=1, num_restarts=2, raw_samples=16,
        )
        next_outputs = torch.cat(
            [task0(next_point), task1(next_point)], dim=-1
        )
    assert next_outputs.shape == (1, 2)
    assert torch.isfinite(surrogate.posterior(next_point).mean).all()
    print(name, next_point, "outputs:", next_outputs,
          "scalarized objective:", next_outputs @ weights)


## 20. KroneckerMultiTaskGPでパレート解を探索する

これまでは $u(x)=0.3f_0(x)+0.7f_1(x)$ の**重み付き単一目的**を最大化しました。ここでは同じ `KroneckerMultiTaskGP` の2出力を**独立した2つの最適化目的**として扱い、パレート解を探索します。`MultiTaskGP` や `ModelListGP` へ切り替える必要はありません。

両目的を最大化するとき、点 $a$ が点 $b$ を**支配**するとは、すべての目的で $a$ が $b$ 以上、かつ少なくとも1目的で厳密に大きいことです。他の点に支配されない点を**非支配解（パレート解）**と呼びます。

`qLogExpectedHypervolumeImprovement`（qLogEHVI）は、参照点 `ref_point` に対して、非支配領域のハイパーボリュームがどれだけ改善しそうかを評価します。重みを固定せずに、目的間のトレードオフを探索できる点がスカラー化との違いです。

この例では `KroneckerMultiTaskGP` が要求するblock-designを維持し、候補 `x` では**両taskを同時に観測**します。

In [ ]:
from botorch.acquisition.multi_objective.logei import (
    qLogExpectedHypervolumeImprovement,
)
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    NondominatedPartitioning,
)
from botorch.utils.multi_objective.pareto import is_non_dominated

pareto_X = train_X_block.clone()
pareto_Y = train_Y_block.clone()

# 参照点は、最大化する両目的で初期観測値より小さく設定します。
ref_point = (pareto_Y.min(dim=0).values - 0.2).tolist()
initial_mask = is_non_dominated(pareto_Y)
print("reference point:", ref_point)
print("initial non-dominated points:", int(initial_mask.sum()))

## 21. Hypervolume改善による候補提案と実測更新

次のコードでは、観測済みの2目的値から非支配分割を作り、qLogEHVIを最大化して次の設計条件を1点提案します。`ref_point` は**両目的とも最大化**する場合の基準点であり、最適化中は固定します。

`q=1` は1回に提案する**設計条件の数**です。`KroneckerMultiTaskGP` の2出力数とは異なります。候補の評価では `task0` と `task1` を両方実行し、block-designの出力行を追加します。

実行時間を抑えるため1反復だけ示します。最適化の性能評価ではなく、**パレート候補の生成からデータ更新までの実装例**です。

In [ ]:
pareto_model = KroneckerMultiTaskGP(train_X=pareto_X, train_Y=pareto_Y)
pareto_model.fit()
pareto_model.eval()

partitioning = NondominatedPartitioning(
    ref_point=torch.tensor(ref_point, dtype=pareto_Y.dtype),
    Y=pareto_Y,
)
pareto_acqf = qLogExpectedHypervolumeImprovement(
    model=pareto_model,
    ref_point=ref_point,
    partitioning=partitioning,
    sampler=SobolQMCNormalSampler(sample_shape=torch.Size([64])),
)
pareto_candidate, pareto_acq_value = optimize_acqf(
    acq_function=pareto_acqf,
    bounds=bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
)
pareto_candidate_Y = torch.cat(
    [task0(pareto_candidate), task1(pareto_candidate)], dim=-1
)
pareto_X = torch.cat([pareto_X, pareto_candidate], dim=0)
pareto_Y = torch.cat([pareto_Y, pareto_candidate_Y], dim=0)
pareto_mask = is_non_dominated(pareto_Y)

assert pareto_candidate.shape == (1, 1)
assert pareto_candidate_Y.shape == (1, 2)
assert pareto_X.shape[0] == pareto_Y.shape[0]
print("Pareto candidate x:", pareto_candidate)
print("observed objectives:", pareto_candidate_Y)
print("log-EHVI acquisition value:", pareto_acq_value)
print("non-dominated observations:", int(pareto_mask.sum()))

## 22. 目的空間でパレートフロントを可視化する

横軸をtask 0、縦軸をtask 1の目的値とします。灰色の点はすべての観測、強調した点は非支配観測です。今回の新しい候補も重ねて表示します。

ここで表示するのは**観測済みデータの非支配集合**であり、未知の真のパレートフロントそのものではありません。候補の追加が必ず非支配点を増やすわけでもありません。

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(pareto_Y[:, 0], pareto_Y[:, 1], alpha=0.5, label="observations")
ax.scatter(
    pareto_Y[pareto_mask, 0],
    pareto_Y[pareto_mask, 1],
    marker="D",
    s=65,
    label="observed Pareto set",
)
ax.scatter(
    pareto_candidate_Y[:, 0],
    pareto_candidate_Y[:, 1],
    marker="*",
    s=150,
    label="new candidate",
)
ax.set_xlabel("task 0 objective (maximize)")
ax.set_ylabel("task 1 objective (maximize)")
ax.legend()
plt.show()